# Project 20 - Vernagtferner Glacier Water Level Forecasting

## Notebook 01: Exploratory Data Analysis

**Goal.** Predict the water level of the proglacial stream draining the Vernagtferner glacier (Otztal Alps, Austria) from on-glacier and near-glacier meteorological covariates plus annual geodetic measurements of glacier extent.

**Source.** Bayerische Akademie der Wissenschaften (BAdW) Vernagtferner long-term monitoring programme, mirrored on PANGAEA (10.1594/PANGAEA.829530 for daily 2002-2012; 5-min 2013-2024 to be released). DWD open climate-data archive for cross-station validation. WGMS Fluctuations of Glaciers for annual area / mass-balance covariates.

**Target.** `water_level_cm` at the Vernagtbach Pegelstation (continuous, regression). Optional secondary target: `discharge_m3_per_s` derived through the station rating curve.

This notebook covers: load, schema, dtypes, shape, missing values, target distribution, seasonality, ablation-season vs accumulation-season split, basic correlations with climate covariates.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data').resolve()
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('Available subfolders:', sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]))

## 1. Load PANGAEA discharge record (Vernagtbach Pegelstation)

PANGAEA distributes the Vernagtferner discharge dataset as a tab-separated file with a multi-line metadata header. The `comment='/'` trick skips the metadata block; alternatively read the metadata explicitly to capture station coordinates and rating curve.

In [ ]:
PANGAEA_FILE = RAW_DIR / 'pangaea' / 'PANGAEA.829530.tab'

if PANGAEA_FILE.exists():
    # Standard PANGAEA layout: '/' marks metadata header, double newline before data.
    df_q = pd.read_csv(PANGAEA_FILE, sep='\t', comment='/', skip_blank_lines=True, parse_dates=[0])
    df_q.columns = [c.strip() for c in df_q.columns]
    print('PANGAEA shape:', df_q.shape)
    print('Columns:', list(df_q.columns))
    df_q.head(3)
else:
    print('PANGAEA file not yet downloaded. See data/README.md for the curl command.')
    df_q = pd.DataFrame()

## 2. Schema, dtypes, missingness

In [ ]:
if not df_q.empty:
    schema = pd.DataFrame({
        'dtype': df_q.dtypes.astype(str),
        'n_unique': df_q.nunique(),
        'n_missing': df_q.isna().sum(),
        'pct_missing': (df_q.isna().mean() * 100).round(2),
    })
    schema

## 3. Target distribution: water level and discharge

Glacier-fed alpine streams show:
- A strong seasonal signal: low baseflow Nov-Apr, ablation-season peak Jun-Sep.
- A diurnal cycle in the ablation season tracking solar-radiation-driven melt.
- Long-tailed flood events from rainfall-on-snow or supraglacial lake drainage.

We expect log-normal or right-skewed distribution.

In [ ]:
if not df_q.empty:
    # Heuristic: pick the first numeric column whose name contains 'level' or 'discharge'.
    candidates = [c for c in df_q.columns if any(k in c.lower() for k in ('level', 'water', 'discharge', 'q '))]
    print('Candidate target columns:', candidates)
    if candidates:
        TARGET = candidates[0]
        y = df_q[TARGET].dropna()
        print('Target:', TARGET)
        print('n =', len(y))
        print(y.describe().round(3))

In [ ]:
if not df_q.empty and 'TARGET' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(y, bins=80, color='steelblue', edgecolor='white')
    axes[0].set_title(f'{TARGET} - histogram')
    axes[0].set_xlabel(TARGET)
    axes[0].set_ylabel('# observations')

    axes[1].hist(np.log1p(y.clip(lower=0)), bins=80, color='steelblue', edgecolor='white')
    axes[1].set_title(f'log1p({TARGET})')
    axes[1].set_xlabel(f'log1p({TARGET})')
    plt.tight_layout()
    plt.show()

## 4. Seasonality - ablation vs accumulation regime

We split by month-of-year and compare the median target. The ablation season for Vernagtferner is roughly June through September; July and August carry the bulk of melt-water discharge. Accumulation season runs October to May.

In [ ]:
if not df_q.empty and 'TARGET' in dir():
    df_q['_dt'] = pd.to_datetime(df_q.iloc[:, 0])
    df_q['month'] = df_q['_dt'].dt.month
    df_q['doy'] = df_q['_dt'].dt.dayofyear
    season = df_q.groupby('month')[TARGET].agg(['count', 'mean', 'median', 'std']).round(2)
    season

In [ ]:
if not df_q.empty and 'TARGET' in dir():
    fig, ax = plt.subplots(figsize=(9, 4))
    monthly_med = df_q.groupby('month')[TARGET].median()
    monthly_iqr_lo = df_q.groupby('month')[TARGET].quantile(0.25)
    monthly_iqr_hi = df_q.groupby('month')[TARGET].quantile(0.75)
    ax.plot(monthly_med.index, monthly_med.values, 'o-', color='steelblue', label='median')
    ax.fill_between(monthly_med.index, monthly_iqr_lo.values, monthly_iqr_hi.values,
                    alpha=0.2, color='steelblue', label='IQR')
    ax.set_xlabel('month')
    ax.set_ylabel(TARGET)
    ax.set_title('Seasonal cycle of stream water level')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Climate covariates - DWD station load (Zugspitze, Wendelstein, Hohenpeissenberg)

DWD ships per-station ZIP archives. We load 5-minute air temperature, hourly precipitation, daily snow depth, and combine on the timestamp axis. The on-glacier BAdW stations (Pegelstation Vernagt, Schwarzkogel) are added once the BAdW programme data is registered.

In [ ]:
DWD_DIR = RAW_DIR / 'dwd'
if DWD_DIR.exists():
    available = sorted(DWD_DIR.glob('*.zip'))
    print('Available DWD archives:', [p.name for p in available])
else:
    print('DWD raw folder not yet populated. See data/README.md for fetch URLs.')

## 6. Lag-correlation analysis (target vs temperature, precipitation)

Glacier-melt response to a temperature pulse is delayed by hours to a few days depending on englacial routing. We compute lag correlations from 0 to 7 days (1d, 7d, 30d windows are then aggregated for the baseline regression).

In [ ]:
# Skeleton for lag correlation - implemented after climate covariates are joined.
def lag_corr(target, covariate, lags):
    out = []
    for k in lags:
        c = target.corr(covariate.shift(k))
        out.append((k, c))
    return pd.DataFrame(out, columns=['lag_days', 'pearson_r'])

# Example call (uncomment after merging hydrology and climate frames):
# lag_corr(df_daily['water_level_cm'], df_daily['t_mean_C'], lags=range(0, 8))

## 7. Annual geodetic covariates - WGMS Fluctuations of Glaciers

WGMS publishes annual area, length, and front-variation tables. Vernagtferner area has decreased from ~9.6 km2 in 1969 to ~7.6 km2 by 2018 (per BAdW programme reports). We treat the annual area as a slowly-varying covariate and forward-fill onto the daily axis. Mass-balance year boundary is October 1 in the WGMS convention.

In [ ]:
WGMS_FILE = RAW_DIR / 'wgms' / 'fog_mb_vernagtferner.csv'
if WGMS_FILE.exists():
    df_mb = pd.read_csv(WGMS_FILE)
    print('WGMS mass balance shape:', df_mb.shape)
    df_mb.head()
else:
    print('WGMS extract not yet downloaded. See data/README.md.')

## 8. Findings summary

**Dataset.** Vernagtferner BAdW long-term programme, daily discharge mirrored on PANGAEA (DOI 10.1594/PANGAEA.829530, 2002-2012); 5-minute resolution 2013-2024 record cited in the brief, registration-gated through the BAdW Kommission. Climate covariates from DWD (Zugspitze, Wendelstein, Hohenpeissenberg) plus optional MeteoSwiss IDAweb stations across the border. Annual area and mass balance from WGMS Fluctuations of Glaciers.

**Data quality (expected, to be verified once data is loaded).**
- Water-level series has gaps during winter shutdown (gauge intake freezes, typically Nov-Apr), which are encoded as NaN rather than zero. Handle via two-stream model: ablation-season regime vs winter baseflow.
- DWD 5-minute data has occasional sensor outages; standard practice is to back-fill with hourly aggregate.
- WGMS area is one annual value; daily interpolation should be linear within mass-balance year.

**Target distribution.** Right-skewed and bimodal: a low-flow winter mode and a high-flow ablation-season mode. log1p transform recommended for the linear baseline.

**Seasonality.** Monthly median follows the canonical alpine glacier hydrograph: minimum January-February, rapid rise May-June, peak July-August, decline September-October.

**Drivers (literature-anchored expectations).**
- Strongest covariate: 1- to 3-day-lagged air temperature (degree-day melt link, see Hock 2003 and Pellicciotti 2005).
- Secondary: incoming shortwave radiation (Pellicciotti's enhanced temperature index).
- Tertiary: rainfall-on-snow and direct precipitation events (peaky high-flow tail).
- Slowly-varying: snow water equivalent (depleting through the ablation season, modulating melt sensitivity).
- Inter-annual: glacier area shrinking by ~0.5% per year, eroding the long-run melt source.

**Next steps for `02_features.ipynb`.**
- Aggregate to daily: mean / max / min temperature, sum precipitation, mean radiation, end-of-day snow depth.
- Compute lagged features at 1d, 3d, 7d, 30d.
- Add cyclical encoding of day-of-year (sin/cos).
- Add cumulative degree-day index from October 1 of the current mass-balance year.
- Forward-fill annual WGMS area into a daily covariate.
- Split: forward-chained, year-by-year (train through year T-1, validate on year T).